In [62]:
from urllib.request import urlopen, Request
from urllib.parse import urlparse
from bs4 import BeautifulSoup
import requests
import time
import random



In [17]:
headers={
    'User-Agent': 
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    }

In [ ]:
url = 'http://en.wikipedia.org/wiki/Kevin_Bacon'
req = Request(url, headers={
    'User-Agent': 
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    })
html = urlopen(req)

soup = BeautifulSoup(html, 'html.parser')

for link in soup.find_all('a'):
    if 'href' in link.attrs:
        print(link.attrs['href'])


In [26]:

# Rewrote the code in the book with bs4 instead of urlopen and regex
# this crawls throgh a given wikipedia url and then finds random article links on the, extracts the href and prints it, repeats the step of fetching any random link from this printed page and prints it again, the continues until the loop becomes false.
# I have give it a maximum page of 5 for testing

def getLinks(articleUrl):
    if articleUrl.startswith('//'):
        articleUrl = 'https:' + articleUrl

    response = requests.get(articleUrl, headers=headers)
    time.sleep(random.uniform(1,3))

    soup = BeautifulSoup(response.text, 'html.parser')
    content = soup.find('div', {'id': 'mw-content-text'})
    return content.find_all('a', {'rel':'mw:WikiLink'})

links = getLinks('https://en.wikipedia.org/wiki/Kevin_Bacon')

count = 0
max_pages = 5

while len(links) > 0 and count < max_pages:
    newArticle = links[random.randint(0, len(links)-1)]['href']
    print(newArticle)
    links = getLinks(newArticle)
    count += 1


https://en.wikipedia.org/wiki/JFK_(film)
https://en.wikipedia.org/wiki/Fargo_(1996_film)
https://en.wikipedia.org/wiki/The_Heap_(Fargo)
https://en.wikipedia.org/wiki/Lorne_Malvo
https://en.wikipedia.org/wiki/Serial_killer


In [ ]:
# to check

pages = set()
def getLinks(pageUrl):
    html = urlopen('http://en.wikipedia.org{}'.format(pageUrl))
    bs = BeautifulSoup(html, 'html.parser')
    try:
        print(bs.h1.get_text())
        print(bs.find(id ='mw-content-text').find_all('p')[0])
        print(bs.find(id='ca-edit').find('span')
            .find('a').attrs['href'])
    except AttributeError:
        print('This page is missing something! Continuing.')
        
    for link in bs.find_all('a', href=re.compile('^(/wiki/)')):
        if 'href' in link.attrs:
            if link.attrs['href'] not in pages:
                #We have encountered a new page
                newPage = link.attrs['href']
                print('-'*20)
                print(newPage)
                pages.add(newPage)
                getLinks(newPage)
getLinks('')

In [ ]:
#  checking status code
url = 'https://wikipedia.org'
response = requests.get(url, headers=headers)

print(response)

<Response [200]>


In [ ]:
# to check from above

pages = set()

def getLinks(pageUrl):
    url = f"https://wikipedia.org{pageUrl}"

    try:
        response = requests.get(url, headers=headers)

        if response.status_code != 200:
            print(f"Skipping {pageUrl}: Status {response.status_code}")

        soup = BeautifulSoup(response.text, 'html.parser')

        try:
            print(f"\n[TITLE]: {soup.h1.get_text()}")
        except:
            print("[INFO]: Missing title element.")

        for link in soup.find_all('a'):
            href = link.get('href')

            if href and href.startswith('/wiki/') and ':' not in href:
                if href not in pages:
                    print(f"Found new link: {href}")
                    pages.add(href)

                    time.sleep(1)

                    getLinks(href)

    except requests.exceptions.RequestException as e:
        print(f"Network error: {e}")

In [31]:
# Recursion - I have no idea what it is at this point
# this was also rewritten without regex and urlopen
# this does something similar to the the code above and instead of picking links at randomit uses a recursion instead of a while loop

pages = set()

def getLinks(pageUrl):
    if len(pages) >= 10:
        return

    if pageUrl.startswith('//'):
        pageUrl = 'https:' + pageUrl

    response = requests.get(pageUrl, headers=headers)
    time.sleep(random.uniform(1,3))
    
    soup = BeautifulSoup(response.text, 'html.parser')
    content = soup.find('div', {'id':'mw-content-text'})

    for link in content.find_all('a', {'rel': 'mw:WikiLink'}):
        if len(pages) >= 10:
            break

        href = link.get('href')
        if href and href not in pages:
            print(href)
            pages.add(href)
            getLinks(href)

getLinks('https://en.wikipedia.org')


https://en.wikipedia.org/wiki/Wikipedia
https://en.wikipedia.org/wiki/Main_Page
https://en.wikipedia.org/wiki/Free_content
https://en.wikipedia.org/wiki/Wikipedia:Free_content
https://en.wikipedia.org/wiki/Sexual_objectification#Free_use
https://en.wikipedia.org/wiki/Sex_object_(disambiguation)
//en.wikipedia.org/wiki/Sexual_objectification
https://en.wikipedia.org/wiki/Wikipedia:WikiProject_Countering_systemic_bias
https://en.wikipedia.org/wiki/Special:EditPage/Sexual_objectification
https://en.wikipedia.org/wiki/Talk:Sexual_objectification


In [ ]:
print(soup.find(id='mw-content-text').find_all('p')[0])
soup.find(id='mwDw').get_text(strip=True)
soup.find(id='ca-edit-sticky-header')
soup.find_all('p')[3].get_text()

<p class="mw-empty-elt" id="mwBQ"><span about="#mwt4" data-mw='{"parts":[{"template":{"target":{"wt":"pp-blp","href":"./Template:Pp-blp"},"params":{"small":{"wt":"yes"}},"i":0}}]}' id="mwBg" typeof="mw:Nowiki mw:Transclusion"></span><meta about="#mwt4" data-mw='{"name":"indicator","attrs":{"name":"pp-default"},"body":{"extsrc":"[[File:Semi-protection-shackle.svg|20px|link=Wikipedia:Protection policy#semi|alt=Page semi-protected|This article is semi-protected to promote compliance with the policy on biographies of living persons]]"}}' id="mwBw" typeof="mw:Extension/indicator"/><link about="#mwt4" href="./Category:Wikipedia_indefinitely_semi-protected_biographies_of_living_people#Kevin%20Bacon" id="mwCA" rel="mw:PageProp/Category"/>
<span about="#mwt6" data-mw='{"parts":[{"template":{"target":{"wt":"Use American English","href":"./Template:Use_American_English"},"params":{"date":{"wt":"April 2023"}},"i":0}}]}' id="mwCQ" typeof="mw:Nowiki mw:Transclusion"></span><link about="#mwt6" href="

'On television, Bacon received aGolden Globe Awardand aScreen Actors Guild Awardfor his role asMichael Stroblin theHBOoriginal filmTaking Chance(2009). He starred in theFoxdrama seriesThe Followingfrom 2013 to 2015. Bacon played the title role inAmazon Prime VideoseriesI Love Dickfrom 2016 to 2017. From 2019 to 2022, he starred in theShowtimeseriesCity on a Hill.[3]'

In [59]:
# Developing the above code further

pages = set()

def getLinks(pageUrl):
    if len(pages) >= 5:
        return

    if pageUrl.startswith('//'):
        pageUrl = 'https:' + pageUrl

    response = requests.get(pageUrl, headers=headers)
    time.sleep(random.uniform(1,3))

    soup = BeautifulSoup(response.text, 'html.parser')

    try:
        print(soup.h1.get_text())
        # print(soup.find(id='mw-content-text').find_all('p')[0])
        print(soup.find_all('p')[1].get_text())
        print(soup.find(id='ca-edit').find('span').find('a').attrs['href'])

    except AttributeError:
        print('This page is missing something! Continuing.')

    content = soup.find('div', {'id': 'mw-content-text'})

    for link in content.find_all('a', {'rel': 'mw:WikiLink'}):
        if len(pages) >= 5:
            break

        href = link.get('href')
        if href and href not in pages:
            print('-' * 20)
            print(href)
            pages.add(href)
            getLinks(href)

getLinks('https://en.wikipedia.org/wiki/Main_Page')

Main Page
July 29
This page is missing something! Continuing.
--------------------
https://en.wikipedia.org/wiki/Wikipedia
Wikipedia
Wikipedia[e] is a free online encyclopedia written and maintained by a community of volunteers, known as Wikipedians, through open collaboration and the wiki software MediaWiki. Founded by Jimmy Wales and Larry Sanger in 2001, Wikipedia has been hosted since 2003 by the Wikimedia Foundation, an American nonprofit organization funded mainly by donations from readers.[1] Wikipedia is the largest and most read reference work in history.[2][3]
This page is missing something! Continuing.
--------------------
https://en.wikipedia.org/wiki/Main_Page
Main Page
July 29
This page is missing something! Continuing.
--------------------
https://en.wikipedia.org/wiki/Free_content
Free content
Free content, libre content, or open content (also called free information, libre information, open information) is any kind of creative work,[1] such as a work of art, a book,[2]

In [74]:
# creating a global soup object once to use in the following functions
def getSoup(url):
    response = requests.get(url, headers=headers)
    time.sleep(random.uniform(1, 3))
    return BeautifulSoup(response.text, 'html.parser')

bs = getSoup('https://en.wikipedia.org/wiki/Kevin_Bacon')

In [ ]:
# Crawling to get all internal links from a web page and storing them in a list
# The purpose of this is given a page content, to return complete list of all internal links as full urls

def getInternalLinks(bs, url):
    netloc = urlparse(url).netloc
    scheme = urlparse(url).scheme
    internalLinks = set()
    for link in bs.find_all('a'):
        if not link.attrs.get('href'):
            continue
        parsed = urlparse(link.attrs['href'])
        if parsed.netloc == '':
            l = f'{scheme}://{netloc}/{link.attrs["href"].strip("/")}'
            internalLinks.add(l)
        elif parsed.netloc == netloc:
            internalLinks.add(link.attrs['href'])
    return list(internalLinks)

getInternalLinks(bs, 'https://en.wikipedia.org/wiki/Kevin_Bacon')

In [ ]:
# urlparse example on a url and the result
url ='https://cat.example/list;meow?breed=siberian#pawsize'
urllib.parse.urlparse(url)

# https = is the scheme (first element of a URL)
# cat.example = is the netloc (sits between the scheme and path)
# /list = is the path (between the netloc and params)
# meow  = is the param (sits between path and query)
# breed=siberian = is the query (between the fragment and params)
# pawsize = is the fragment (last element of a URL)

ParseResult(scheme='https', netloc='cat.example', path='/list', params='meow', query='breed=siberian', fragment='pawsize')

In [ ]:
# modern urlsplit on a url and the result
url ='https://cat.example/list;meow?breed=siberian#pawsize'
urllib.parse.urlsplit(url)


SplitResult(scheme='https', netloc='cat.example', path='/list;meow', query='breed=siberian', fragment='pawsize')

In [ ]:
# Crawling to get all external links from a web page and storing them in a list
# The purpose of this is given a page content, to return complete list of all external links which will obviously be full urls

def getExternalLinks(bs, url):
    internal_netloc = urlparse(url).netloc
    externalLinks = set()
    for link in bs.find_all('a'):
        if not link.attrs.get('href'):
            continue
        parsed = urlparse(link.attrs['href'])
        if parsed.netloc != '' and parsed.netloc != internal_netloc:
            externalLinks.add(link.attrs['href'])
    return list(externalLinks)

getExternalLinks(bs, 'https://en.wikipedia.org/wiki/Kevin_Bacon')

In [82]:
# for this, go through a page and find random external links, if a page doesnt any, dig dipper into random internal links until you find an external link

def getRandomExternalLink(startingPage, depth=0, max_depth=5):
    if depth >= max_depth:
        print('Max search depth reached, giving up on the branch.')
        return None
    
    bs = getSoup(startingPage)
    externalLinks = getExternalLinks(bs, startingPage)
    if not len(externalLinks):
        print('No external links, looking around the site for one')
        internalLinks = getInternalLinks(bs, startingPage)
        return getRandomExternalLink(random.choice(internalLinks), depth + 1, max_depth)
    else:
        return random.choice(externalLinks)

getRandomExternalLink('https://en.wikipedia.org/wiki/Kevin_Bacon')

'https://web.archive.org/web/20100720060214/http://www.forcesofgeek.com/2010/07/kevin-bacon-playing-sebastian-shaw-in-x.html'

In [83]:
def followExternalOnly(startingSite, count=0, max_hops=5):
    if count >= max_hops:
        print('Reached hop limit, stopping.')
        return
    
    externalLink = getRandomExternalLink(startingSite)
    if externalLink is None:
        print('Counld not find an external link, stopping.')
        return

    print(f'Random external link is: {externalLink}')
    followExternalOnly(externalLink, count + 1, max_hops)

followExternalOnly('https://en.wikipedia.org/wiki/Kevin_Bacon')

Random external link is: https://web.archive.org/web/20130405182304/http://www.drawtheline.org/watch-stuff/
No external links, looking around the site for one
No external links, looking around the site for one
No external links, looking around the site for one
No external links, looking around the site for one
No external links, looking around the site for one
Max search depth reached, giving up on the branch.
Counld not find an external link, stopping.


In [ ]:
allExtLinks = {}
allIntLinks = []

def getAllExternalLinks(url, max_pages=20):
    if url not in allIntLinks:
        allIntLinks.append(url)

    if len(allIntLinks) >= max_pages:
        return
    
    bs = getSoup(url)
    internalLinks = getInternalLinks(bs, url)
    externalLinks = getExternalLinks(bs, url)

    for link in externalLinks:
        if link not in allExtLinks:
            # allExtLinks.append(link)
            # print(link)
            allExtLinks[link] = url
            print(f"{link} (found on {url})")
    for link in internalLinks:
        if len(allIntLinks) >= max_pages:
            break
        if link not in allIntLinks:
            allIntLinks.append(link)
            print(f"Int link: {link}, found on {url}")
            getAllExternalLinks(link, max_pages)


getAllExternalLinks('https://www.litfinadvisors.com/')

https://app.instapage.com/route/24144507/?url=www.litfinadvisors.com (found on https://www.litfinadvisors.com/)
https://v.fastcdn.co/u/cf12b1b4/65663089-0-Litigation-Funders-B.pdf (found on https://www.litfinadvisors.com/)
https://v.fastcdn.co/u/cf12b1b4/65663095-0-Member-Spotlight-Reb.pdf (found on https://www.litfinadvisors.com/)
https://v.fastcdn.co/u/cf12b1b4/65663097-0-You-Probably-Cant-Bu.pdf (found on https://www.litfinadvisors.com/)
https://v.fastcdn.co/u/cf12b1b4/65663105-0-Litigation-Funders-F.pdf (found on https://www.litfinadvisors.com/)
https://v.fastcdn.co/u/cf12b1b4/65663068-0-How-Litigation-Finan.pdf (found on https://www.litfinadvisors.com/)
https://v.fastcdn.co/u/cf12b1b4/65663078-0-Bloomberg-Law---Liti.pdf (found on https://www.litfinadvisors.com/)
https://v.fastcdn.co/u/cf12b1b4/65663094-0-Opioids-Lawyers-Offe.pdf (found on https://www.litfinadvisors.com/)
https://v.fastcdn.co/u/cf12b1b4/65663083-0-Clients-Embrace-Liti.pdf (found on https://www.litfinadvisors.com/)
